# CERIDWEN
## from little tiny baby steps to stuff that could actually work for a paper in just 30 min! 

**Part A** we will do a lil test all together, with one model that I specified, so this will only encompass one model
which we will fit to some some mock photometry. 


But you probably already have a specific science case 
in your head which just can not wait!
So **Part B** will be a little bit more free, where
 you can choose the kind of observation
and populate it with real data. Then you can choose
 the libraries you want to use 
the *grid* (SSPData), frankenstein a forward model,
 fit it and post-process it.
Maybe this will happen fast enough for someone to
 present at the end! (So lock in please.)


Everything runs on the GPU (Apple GPU in float32, or NVIDIA in float64).

In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp

import ceridwen
from ceridwen import (SSPData, CSPBasis, SedModel, fitSED, PostProcess, Cosmology,
                      Kinematics, Instrument, TIED, load_postprocess)
from ceridwen.observation import Photometry, Spectrum, Lines
from ceridwen.observation.filters import list_available_filters
from ceridwen.model import logsfr_ratios_to_sfh
from ceridwen.priors import Uniform, ClippedNormal, StudentT, LogUniform
from ceridwen.ssps import fetch_grid, available_grids
from ceridwen.precision import precision_name
from ceridwen.plotting import summary_figure, corner_figure

print("ceridwen  :", ceridwen.__version__)
print("device    :", jax.devices())
print("precision :", precision_name())
print("SPS_HOME  :", os.environ.get("SPS_HOME"))
_v = tuple(int(x) for x in ceridwen.__version__.split(".")[:3])
if _v < (1, 0, 14):
    raise RuntimeError(f"this notebook needs ceridwen >= 1.0.14 (model.display() with emission "
                       f"lines, CSPBasis(fixed=...)), found {ceridwen.__version__}: "
                       f"pip install -U ceridwen")
if jax.default_backend() == "cpu":
    raise RuntimeError("JAX runs on the CPU; this tutorial needs the GPU (see the prep email)")

NJY = 3631e9                      # nJy per AB maggie

# Part A: the simplest fit
We fill fit in 5 steps. We will choose the stellar libraries we use, the model, make the mock data, fit it and plot the results. 
The galaxy is a mock at z = 0.1 observed in twelve bands from GALEX to WISE, made with the same model, so we know the answer.
(and do not have to worry about mode misspecification, which you should all remember from my brilliant Journal Club presentation yesterday)

### A1. The stellar population grid

`fetch_grid` downloads a published grid and returns its path so you do not have to download it next time you use this exact code. We use it to build our SSP = Single Stellar Population object, which is basically a grid of stellar spectra over age and metallicity.
`display()` prints the details you should check before deciding if this is the right grid for you. It is also a great function in case you forgot which one you loaded.


In [ ]:
ssp = SSPData.load(fetch_grid("mist_miles_chab"))
ssp.display()

### A2. The model

Next we need to decide which of the physics are relevant for our problem and which of them we want to include/exclude. For that we build a 
`CSPBasis`, CSP = Composite Stellar Population, and turn possible ingredients on or off. If we want to fit for something it needs 
to be included here.

`lookback_time` holds the SFH nodes in Gyr, index 0 = today. Here: a 6-bin step SFH (constant SFR within each bin, 7 nodes), one metallicity, diffuse dust and nebular emission (continuum and lines,
from CLOUDY grids in `$SPS_HOME`).

Switching the nebular model on adds two parameters to the CSP: the gas-phase metallicity
`gas_logz` and the ionisation parameter `gas_logu` (both log10). We fit them, so they get a prior
below, like every other free parameter. (To keep the lines but not fit these two, fix them when
you build the CSP: `CSPBasis(..., fixed={"gas_logz": -0.3, "gas_logu": -2.5})`. A fixed parameter
needs no prior.)

In [ ]:
cosmo = Cosmology.planck18()
Z_A = 0.1
lookback_A = [0.0, 0.01, 0.03, 0.1, 0.5, 2.0, float(cosmo.age(Z_A))]

csp_A = CSPBasis(ssp, lookback_time=jnp.asarray(lookback_A), cosmo=cosmo,
                 zh_const=True, sfh_interp="step",
                 add_dust=False, add_diffuse_dust=True, add_neb=True,
                 verbose=False)
print("CSP parameters:", csp_A.param_names)



In [ ]:
csp_A

We do not want to fit all of these parameters directly, since it can be very hard to define priors for something like an absolute star formation rate, so instead we define 'transforms'. These take basically anything as an input, and are basically just a reparametrisation. In this example, we want to fit for the ratio between sfh bins, so we need to write a function that translates them back to the SFRs that the CSPBasis expects:

In [ ]:
times_A = np.array(csp_A.sfh_times)
TRANSFORMS_A = {"sfh": lambda th, _t=times_A: logsfr_ratios_to_sfh(th["logsfr_ratios"], sfh_times_yr=_t)}

The model also includes the priors which we use for the fit. Each free parameter needs a prior:

In [ ]:
PRIORS_A = {"logsfr_ratios": StudentT(mean=0.0, scale=0.3, df=2.0),
            "logmass": Uniform(low=8.0, high=12.0),
            "logzsol": Uniform(low=-2.0, high=0.2),
            "diffuse_tau_kc": ClippedNormal(mean=0.3, sigma=1.0, low=0.0, high=4.0),
            "diffuse_dust_index": Uniform(low=-1.0, high=0.4),
            "gas_logz": Uniform(low=-1.5, high=0.2),       # gas-phase log(Z/Z_sun)
            "gas_logu": Uniform(low=-3.5, high=-1.0)}      # log ionisation parameter


And because it is a mock test, we also define the truths:

In [ ]:
TRUTH_A = {"logsfr_ratios": jnp.array([0.2, 0.3, 0.1, -0.3, -0.5]), "logmass": jnp.array([10.3]),
           "logzsol": jnp.array([-0.3]), "diffuse_tau_kc": jnp.array([0.6]),
           "diffuse_dust_index": jnp.array([-0.5]),
           "gas_logz": jnp.array([-0.3]), "gas_logu": jnp.array([-2.5])}

### A3. The data
Next we are generating the data:
Photometry is in AB maggies (1 maggie = 3631 Jy). We push the truth through the model and add
noise at S/N = 20.

We also measure five emission lines. Line fluxes are integrated, observed-frame fluxes in
erg s$^{-1}$ cm$^{-2}$. CERIDWEN identifies a
line by its row in the nebular grid and its vacuum rest wavelength; `csp_A.neb.nebem_line_pos` and
`csp_A.neb.nebem_line_names` list them.

In [ ]:
FILTERS_A = ["galex_FUV", "galex_NUV", "sdss_u0", "sdss_g0", "sdss_r0", "sdss_i0", "sdss_z0",
             "twomass_J", "twomass_H", "twomass_Ks", "wise_w1", "wise_w2"]
n_A = len(FILTERS_A)

phot = Photometry(filters=FILTERS_A, 
           flux=np.zeros(n_A),
           uncertainty=np.ones(n_A), 
           name="phot")

# five lines by their vacuum rest wavelength [A]: the nearest row of the nebular grid
LINES_WANT = {"H beta": 4862.7, "[O III] 5007": 5008.2, "H alpha": 6564.6,
              "[N II] 6584": 6585.3, "[S II] 6717": 6718.3}



LINES_WANT = {"H beta": 4862.7, "[O III] 5007": 5008.2, "H alpha": 6564.6,
              "[N II] 6584": 6585.3, "[S II] 6717": 6718.3}     # vacuum rest wavelengths [A]
pos = np.asarray(csp_A.neb.nebem_line_pos)
ROWS_A = [int(np.argmin(np.abs(pos - w))) for w in LINES_WANT.values()]   # ceridwen re-checks these
n_L = len(ROWS_A)

def make_lines_A(flux, err):
    return Lines(line_ind=ROWS_A, line_names=list(LINES_WANT), wavelength=[float(pos[r]) for r in ROWS_A],
                 flux=np.asarray(flux), uncertainty=np.asarray(err), name="lines")


lines = make_lines_A(np.zeros(n_L), np.ones(n_L))

obs = [phot, lines]

phot.display()


In [ ]:
csp_A

In [ ]:
lines.display()

The photometry and the lines are still empty at the moment, but we only need the filter information that defines how the photometry should look like for the next step. As soon as we have the observation strucuture, we have all three ingredients for the SedModel, which takes the physical model (CSPBasis) and returns to us what we would expect the observation(s) to look like.

In [ ]:
truth_model_A = SedModel(csp_A,
                         obs,
                         priors=PRIORS_A, 
                         transforms=TRANSFORMS_A, 
                         free_param_init=TRUTH_A, 
                         zred=Z_A)

now we can use the SED model to predict the mock data by running SedModel.predict() while giving it the parameter vector we defined earlier, TRUTH_A

In [ ]:
pred_A = truth_model_A.predict(TRUTH_A)
true_A = np.asarray(pred_A["phot"])
rng = np.random.default_rng(1)
err_A = true_A / 20.0

phot_A = Photometry(filters=FILTERS_A, flux=true_A + err_A * rng.standard_normal(n_A),
                    uncertainty=err_A, name="phot")

true_L = np.asarray(pred_A["lines"])           # erg/s/cm^2
err_L = true_L / 10.0                          # S/N = 10 per line
lines_A = make_lines_A(true_L + err_L * rng.standard_normal(n_L), err_L)
for label, f, e in zip(LINES_WANT, true_L, err_L):
    print(f"  {label:14s} {f:.3e} +- {e:.1e} erg/s/cm^2")


Now we can build a new model, which we give the now filled photometry and line objects (the rest does not change.) Also we need to initialise all free parameters that the csp does not know already, so it knows which shape the inputs will have when a transform happens. (`logmass` needs only its prior: ceridwen starts it at the prior's median.) Then we can look at the now complete model.

In [ ]:
model_A = SedModel(csp_A, [phot_A, lines_A], priors=PRIORS_A, transforms=TRANSFORMS_A,
                   free_param_init={"logsfr_ratios": jnp.zeros(5)},
                   zred=Z_A)
model_A.display()

### A4. The fit

Nested sampling. `num_live` live points; `num_delete` are replaced per step, in parallel on
the GPU; `num_inner_steps` slice steps per replacement. These are demo settings.

In [ ]:
result_A = fitSED(model_A, sampler="nested", rng_key=jax.random.PRNGKey(0),
                  sampler_kwargs={"num_live": 150, "num_delete": 30, "num_inner_steps": 30}, #for a proper result: sampler_kwargs={"num_live": 500, "num_delete": 100, "num_inner_steps": 60, "logZ_tol": -5.0},
                  output_dir="fit_partA")
print(f"ln Z = {result_A.log_evidence:.2f} +- {result_A.log_evidence_err:.2f}, "
      f"{result_A.wall_time_s:.0f} s")

### A5. The result

`PostProcess` turns the weighted samples into equal-weight draws and pushes each through the
model: parameters, derived quantities, posterior-predictive data.

In [ ]:
out_A = PostProcess(model_A, result_A, n_samples=1000).run()
truths_A = {k: (float(v[0]) if v.size == 1 else np.asarray(v)) for k, v in TRUTH_A.items()}


def crop_sed(fig, model):
    # summary_figure pads the data range by 25 % in log wavelength; crop the SED panel (and the
    # residuals that share its x axis) to the filter edges and the spectrum ends [micron]
    lo, hi = [], []
    for o in model.observations:
        if isinstance(o, Photometry):
            for f in o.filters:
                w = f.wavelength[f.transmission > 0.01 * f.transmission.max()]
                lo.append(w.min()); hi.append(w.max())
        elif isinstance(o, Spectrum):
            lo.append(float(np.min(o.wavelength))); hi.append(float(np.max(o.wavelength)))
    if lo:
        fig.axes[0].set_xlim(min(lo) / 1e4, max(hi) / 1e4)
    return fig


crop_sed(summary_figure(out_A, model_A, title="Part A: mock galaxy", truths=truths_A), model_A)
plt.show()

# Part B: now it's your turn

You build the model yourself. In order:

1. **B1** choose the stellar population grid
2. **B2** read the list of knobs
3. **B3** set the knobs and build the physics (`CSPBasis`)
4. **B4** define your observations (photometry, emission lines, a spectrum)
5. **B5** priors: what is free, what is fixed
6. **B6** the data and the model, then `model.display()`
7. **B7** fit
8. **B8** post-process

Run the cells in order. With `DATA_SOURCE = "mock"` the notebook makes mock data for every
observation you switch on, so it runs as it is; switch to `"mine"` and fill in B4 for real data.

## B1. Choose the stellar population grid

The three published grids (Zenodo, doi:10.5281/zenodo.22937956), downloaded on first use into
`~/.ceridwen/grids`:

| name | isochrones + spectra | IMF | size | what it is for |
|---|---|---|---|---|
| `mist_miles_chab` | MIST + MILES | Chabrier | 67 MB | the default; MILES gives a good optical resolution (FWHM 2.54 Å, 3525–7500 Å rest) |
| `mist_bpass_v2` | BPASS v2 binary populations | Chabrier | 62 MB | binary stars: harder ionising spectra, matters for young, metal-poor galaxies |
| `amist_c3k_hr_krou_afe` | aMIST + C3K, five [α/Fe] planes (−0.2 … +0.6) | Kroupa | 612 MB | [α/Fe] as a fitted parameter; **no nebular model** (no emission lines, no `Lines`), continuum only, so quiescent galaxies |

Notes:

- On MIST grids `logzsol` is [Fe/H]; on every grid it is log10(Z/Z_sun) with the grid's own Z_sun.
- The α grid has no surviving-mass table: fit and post-process it with `mfrac=False` (we do that below).
- The α grid uses `SSPDataAfe` and `CSPBasis_afe` instead of `SSPData` and `CSPBasis`.

In [ ]:
print(available_grids(published_only=True))

GRID = "mist_miles_chab"          # "mist_miles_chab" | "mist_bpass_v2" | "amist_c3k_hr_krou_afe"

IS_AFE = GRID == "amist_c3k_hr_krou_afe"
if IS_AFE:
    from ceridwen.ssps import SSPDataAfe
    from ceridwen.csp import CSPBasis_afe
    ssp_B = SSPDataAfe.load(fetch_grid(GRID))
else:
    ssp_B = SSPData.load(fetch_grid(GRID))
ssp_B.display()

## B2. The knobs

Everything you can switch in the next cell. "Adds" lists the parameters a knob brings in; each
of them gets a prior in B5 or is held fixed at its default.

**Redshift and star formation history**

| knob | what it does | adds |
|---|---|---|
| `ZRED` | the galaxy's redshift | |
| `FIT_ZRED` | sample the redshift (bounded prior around `ZRED`); the SFH grid then stretches with the age of the Universe (`track_zred_age=True`) | `zred` |
| `N_OLD_BINS` | number of log-spaced SFH bins beyond 300 Myr (the young nodes are 0, 10, 50, 100, 300 Myr) | longer `logsfr_ratios` |
| `SFH_INTERP` | `"step"` (constant SFR within a bin) or `"linear"` (linear between nodes) | |
| `SFH_PRIOR_SCALE` | width of the Student-t (df = 2) continuity prior on adjacent log SFR ratios: 0.3 smooth, 1.0 bursty | |

**Metallicity**

| knob | what it does | adds |
|---|---|---|
| `METALLICITY_HISTORY` | False: one `logzsol` for all stars; True: one per SFH bin, constant within the bin | `logzsol` or `logzsol_bins` |
| `GAS_TIED` | gas-phase metallicity = stellar metallicity (Prospector's choice) | removes `gas_logz` |
| `FIT_AFE` | sample [α/Fe] (α grid only) | `afe` |

**Dust**

| knob | what it does | adds |
|---|---|---|
| `DIFFUSE_DUST` | a dust screen on all stars | `diffuse_tau_*` and the law's other parameters |
| `DIFFUSE_LAW` | `"kriek_conroy"` (Calzetti + bump + free slope), `"calzetti"`, `"smc"`, `"lmc"`, `"reddy15"`, `"gordon03_smcbar"`, `"cardelli"`, `"powerlaw"`, `"noll"`, `"conroy"`, ... | |
| `BIRTH_CLOUD_DUST` | extra dust on stars younger than ~10 Myr (power law) | `tau_pow`, `alpha_pow` |
| `DUST_EMISSION` | dust re-emission in the IR, energy balance (DL07 templates from `$SPS_HOME`) | `duste_qpah`, `duste_umin`, `duste_gamma` |

**Gas, escape, IGM, AGN**

| knob | what it does | adds |
|---|---|---|
| `NEBULAR` | CLOUDY nebular continuum and emission lines (`$SPS_HOME`; not on the α grid) | `gas_logu` (+ `gas_logz`) |
| `FIT_FESC` | a fraction of light that escapes dust and gas (`fesc_geometry` `"runaway_bc"` or `"picket"`) | `frac_obrun` |
| `IGM` | Madau (1995) IGM absorption, scales with z | |
| `FIT_IGM_FACTOR` | scale the IGM optical depth | `igm_factor` |
| `AGN` | torus + accretion disc + narrow lines (`AGNConfig()`, see `docs/agn.md`) | `log_fagn`, `agn_tau`, `log_fagn_bbb` (+ fixed others) |

**Kinematics (velocity dispersions, km/s)**

| knob | what it does | adds |
|---|---|---|
| `SIGMA_GAL` | stellar dispersion: a number (fixed) or `"free"` | `sigma_gal` |
| `SIGMA_GAS` | gas dispersion: `"tied"` to the stars, a number, or `"free"` | `sigma_gas` |
| `BROADEN_PHOTOMETRY` | apply the stellar dispersion to the spectrum the filters see (tiny effect on broad bands) | |

**Observations and their nuisance terms**

| knob | what it does | adds |
|---|---|---|
| `USE_PHOTOMETRY`, `USE_LINES`, `USE_SPECTRUM` | the data you want to fit | |
| `PHOT_OUTLIERS` | outlier mixture for bad bands (Hogg+10; keep off for few bands) | `f_outlier_phot` |
| `ELINE_SCALING` | one factor on all line fluxes (slit losses relative to the photometry) | `eline_scaling` |
| `SPEC_R`, `FIT_LSF_SCALE` | spectrograph resolving power (λ/FWHM); sample a factor on the LSF width | `lsf_scale` |
| `SPEC_CALIBRATION` | `"none"`, `"scaling"`, `"scaling+legendre"`, `"profile"`, `"marginalize"` (see B4) | `spectrum_scaling`, `spectrum_calib` |
| `SPEC_NOISE_FLOOR` | fractional floor on the spectrum errors | |
| `SPEC_JITTER` | extra white noise in data units | `log_jitter_spec` |
| `SPEC_OUTLIERS` | outlier mixture for bad pixels | `f_outlier_spec` |
| `MARGINALIZE_ELINES` | integrate the line fluxes in the spectrum out analytically (needs float64: not on the Apple GPU) | |

**Speed**

| knob | what it does |
|---|---|
| `WAVELENGTH_SUPPORT` | synthesise only the model pixels the data read (faster; not byte-identical to the full grid) |

## B3. Set the knobs and build the physics

Change the values in the first block, then run the cell. It prints every parameter the CSP has.

In [ ]:
# ------------------------------- KNOBS -------------------------------------------------
ZRED                = 3.0
FIT_ZRED            = False
N_OLD_BINS          = 3
SFH_INTERP          = "step"
SFH_PRIOR_SCALE     = 0.3

METALLICITY_HISTORY = False
GAS_TIED            = True
FIT_AFE             = True            # only used on the alpha grid

DIFFUSE_DUST        = True
DIFFUSE_LAW         = "kriek_conroy"
BIRTH_CLOUD_DUST    = True
DUST_EMISSION       = False

NEBULAR             = True            # forced off on the alpha grid
FIT_FESC            = False
FESC_GEOMETRY       = "runaway_bc"    # or "picket" (needs NEBULAR)
IGM                 = True
FIT_IGM_FACTOR      = False
AGN                 = False

SIGMA_GAL           = 150.0           # a number (km/s) or "free"
SIGMA_GAS           = "tied"          # "tied", a number, or "free"
BROADEN_PHOTOMETRY  = True

WAVELENGTH_SUPPORT  = False
# ----------------------------------------------------------------------------------------

if IS_AFE and NEBULAR:
    print("alpha grid: no nebular model, so NEBULAR, GAS_TIED and Lines are switched off")
    NEBULAR = False
if not NEBULAR:
    GAS_TIED = False
SPS_HOME = os.environ.get("SPS_HOME")
if (NEBULAR or DUST_EMISSION or AGN) and SPS_HOME is None:
    raise RuntimeError("NEBULAR / DUST_EMISSION / AGN read data files from $SPS_HOME, which is not set")


def lookback_grid(zred, n_old):
    # Young nodes 0, 10, 50, 100, 300 Myr (those younger than the Universe), then n_old bins
    # log-spaced to the age of the Universe at zred.  Gyr, index 0 = today.
    age = float(cosmo.age(zred))
    nodes = [t for t in (0.0, 0.01, 0.05, 0.1, 0.3) if t < 0.8 * age]
    nodes += list(np.geomspace(nodes[-1], age, n_old + 1)[1:])
    return np.asarray(nodes)


LOOKBACK_B = lookback_grid(ZRED, N_OLD_BINS)
N_RATIOS = LOOKBACK_B.size - (2 if SFH_INTERP == "step" else 1)   # step: one SFR per bin

csp_kw = dict(lookback_time=jnp.asarray(LOOKBACK_B), cosmo=cosmo,
              zh_const=not METALLICITY_HISTORY, sfh_interp=SFH_INTERP,
              add_dust=BIRTH_CLOUD_DUST, add_diffuse_dust=DIFFUSE_DUST, diffuse_law=DIFFUSE_LAW,
              add_dust_emission=DUST_EMISSION, add_igm=IGM,
              track_zred_age=FIT_ZRED, wavelength_support=WAVELENGTH_SUPPORT,
              sps_home=SPS_HOME, verbose=False)
if AGN:
    from ceridwen.agn import AGNConfig
    csp_kw["agn"] = AGNConfig()
if IS_AFE:
    csp_B = CSPBasis_afe(ssp_B, **csp_kw)
else:
    csp_B = CSPBasis(ssp_B, add_neb=NEBULAR, gas_tied=GAS_TIED, fesc_geometry=FESC_GEOMETRY, **csp_kw)

print(f"SFH nodes [Gyr]: {np.round(LOOKBACK_B, 4)}  ({N_RATIOS} free log SFR ratios)")
print("CSP parameters :", csp_B.param_names)

**Finding emission lines.** CERIDWEN identifies a line by its row in the nebular grid (FSPS's
`emlines_info.dat`) and its vacuum rest wavelength. `find_lines("O III")` prints the rows that
match a name.

In [ ]:
def find_lines(pattern=""):
    # (row, rest wavelength [A], FSPS name) of the nebular grid lines whose name contains pattern
    if not NEBULAR:
        print("no nebular model"); return []
    pos = np.asarray(csp_B.neb.nebem_line_pos)
    names = [n or "" for n in (csp_B.neb.nebem_line_names or [None] * pos.size)]   # None: unmatched row
    rows = [(i, float(pos[i]), names[i]) for i in range(pos.size) if pattern.lower() in names[i].lower()]
    for r in rows:
        print(f"  row {r[0]:3d}  {r[1]:10.2f} A  {r[2]}")
    return rows


find_lines("H ")       # hydrogen lines; try "O III", "N II", "S II", "O II", "Ne III"

## B4. Your observations

**Units CERIDWEN expects**

| observation | data | uncertainty | wavelengths |
|---|---|---|---|
| `Photometry` | AB maggies: nJy / 3.631×10¹² | 1σ, same unit, one per band | filter names (`list_available_filters("jwst")`) |
| `Lines` | integrated flux, erg s⁻¹ cm⁻² (observed, not dust-corrected) | 1σ, same unit | vacuum **rest** Å + grid row (`find_lines`) |
| `Spectrum` | F_ν in erg s⁻¹ cm⁻² Hz⁻¹ (1 nJy = 10⁻³² of these) | 1σ per pixel, same unit | vacuum **observed** Å, increasing |

The uncertainty always goes in `uncertainty=`, as a 1σ error in the same unit as the data.
Keep measured fluxes and errors for non-detections (negative values are fine); `upper_limit=`
(a bool array on `Photometry` and `Lines`) instead gives a one-sided penalty.

**Options per observation**

- `mask=` (bool, True = used) on all three; `spec.mask_lines(rest_waves, dv=1000, zred=z)` masks
  ± dv km/s around lines in a spectrum.
- `Photometry`: `upper_limit=`; outlier fraction `f_outlier_phot` (knob below).
- `Lines`: `upper_limit=`; `components=` for unresolved blends (rest wavelengths summed into one
  measurement); `eline_scaling` (one factor on all lines, e.g. slit losses).
- `Spectrum`: `instrument=` the LSF (`Instrument.R_fwhm(R)`, `.sigma_kms(v)`, `.fwhm_aa(w)`, or an
  array with `wave=`), `noise_floor=` (σ² → σ² + (f·model)²), `calibration=` (fixed vector),
  `sky=`, and the calibration modes:
    - `"scaling"`: sample `spectrum_scaling`, one factor on the whole spectrum
    - `"scaling+legendre"`: plus `spectrum_calib`, a Legendre polynomial shape
    - `"profile"`: `polynomial_order=M`, coefficients solved at every call (no extra dimensions)
    - `"marginalize"`: `polynomial_order=M, polynomial_mode="marginalize", polynomial_prior_sigma=s`,
      coefficients integrated out (calibration uncertainty propagated)
- Noise terms (sampled, any observation kind `phot` / `spec` / `lines`): `log_err_scale_<kind>`,
  `log_jitter_<kind>`, `log_f_calib_<kind>`, `log_f_data_<kind>`, and outliers
  `f_outlier_<kind>` with `nsigma_outlier_<kind>` (default 50). See `docs/noise_calibration.md`.

In [ ]:
# ------------------------------- OBSERVATION KNOBS -------------------------------------
DATA_SOURCE        = "mock"           # "mock" or "mine"

USE_PHOTOMETRY     = True
PHOT_OUTLIERS      = False

USE_LINES          = True
LINE_ROWS          = []               # grid rows (find_lines); [] = Hbeta, [O III] 5007, Halpha
ELINE_SCALING      = False

USE_SPECTRUM       = True
SPEC_R             = 1000.0           # resolving power lambda / FWHM
FIT_LSF_SCALE      = False
SPEC_CALIBRATION   = "scaling"        # "none" | "scaling" | "scaling+legendre" | "profile" | "marginalize"
SPEC_POLY_ORDER    = 3
SPEC_NOISE_FLOOR   = 0.0
SPEC_JITTER        = False
SPEC_OUTLIERS      = False
MARGINALIZE_ELINES = False
# ----------------------------------------------------------------------------------------

if IS_AFE and USE_LINES:
    print("alpha grid: no nebular lines, USE_LINES switched off"); USE_LINES = False
if not NEBULAR and USE_LINES:
    print("NEBULAR is off: USE_LINES switched off"); USE_LINES = False
if SPEC_CALIBRATION == "marginalize" and (SPEC_OUTLIERS or MARGINALIZE_ELINES):
    raise ValueError("the marginalised polynomial cannot be combined with spectrum outliers or "
                     "marginalize_elines on the same spectrum")
if MARGINALIZE_ELINES and precision_name() != "float64":
    raise ValueError("marginalize_elines needs float64 (NVIDIA or CPU), not the Apple GPU")

**Your data.** With `DATA_SOURCE = "mine"`, fill in the arrays below (only for the observations
you switched on). With `"mock"` this cell is skipped and B6 makes the data.

In [ ]:
if DATA_SOURCE == "mine":
    # Photometry: filter names, fluxes and 1-sigma errors in nJy
    MY_FILTERS  = ["jwst_f090w", "jwst_f115w", "jwst_f150w", "jwst_f200w",
                   "jwst_f277w", "jwst_f356w", "jwst_f444w"]
    MY_PHOT_NJY = np.array([...])                  # nJy
    MY_PHOT_ERR = np.array([...])                  # nJy
    # Lines: grid rows from find_lines, integrated fluxes and errors in erg/s/cm^2
    MY_LINE_ROWS = [...]
    MY_LINE_FLUX = np.array([...]) * 1e-20         # e.g. a catalogue in 1e-20 erg/s/cm^2
    MY_LINE_ERR  = np.array([...]) * 1e-20
    # Spectrum: observed vacuum Angstrom, F_nu in nJy (converted to cgs below)
    MY_SPEC_WAVE = np.array([...])
    MY_SPEC_NJY  = np.array([...])
    MY_SPEC_ERR  = np.array([...])

**B4b. Build the observation objects.** One function per kind, so the same code makes the
mock containers and your real ones.

In [ ]:
NJY_TO_CGS = 1e-32                     # erg/s/cm^2/Hz per nJy

DEFAULT_FILTERS = ["acs_wfc_f435w", "acs_wfc_f606w", "acs_wfc_f814w",
                   "jwst_f090w", "jwst_f115w", "jwst_f150w", "jwst_f200w", "jwst_f277w",
                   "jwst_f335m", "jwst_f356w", "jwst_f410m", "jwst_f444w"]


def default_line_rows():
    want = (4862.7, 5008.2, 6564.6)                # Hbeta, [O III] 5007, Halpha (vacuum rest)
    pos = np.asarray(csp_B.neb.nebem_line_pos)
    return [int(np.argmin(np.abs(pos - w))) for w in want]


def make_photometry(filters, flux_maggies, err_maggies):
    return Photometry(filters=filters, flux=np.asarray(flux_maggies),
                      uncertainty=np.asarray(err_maggies), name="phot")


def make_lines(rows, flux, err):
    pos = np.asarray(csp_B.neb.nebem_line_pos)
    names = csp_B.neb.nebem_line_names or [None] * pos.size
    return Lines(line_ind=list(rows), line_names=[names[r] or f"{pos[r]:.0f}" for r in rows],
                 wavelength=[float(pos[r]) for r in rows],
                 flux=np.asarray(flux), uncertainty=np.asarray(err), name="lines")


def make_spectrum(wave_obs, flux_cgs, err_cgs):
    scale = "lsf_scale" if FIT_LSF_SCALE else 1.0
    kw = dict(wavelength=np.asarray(wave_obs), flux=np.asarray(flux_cgs),
              uncertainty=np.asarray(err_cgs), name="spec",
              instrument=Instrument.R_fwhm(SPEC_R, scale=scale),
              noise_floor=SPEC_NOISE_FLOOR, marginalize_elines=MARGINALIZE_ELINES)
    if SPEC_CALIBRATION == "profile":
        kw["polynomial_order"] = SPEC_POLY_ORDER
    elif SPEC_CALIBRATION == "marginalize":
        kw.update(polynomial_order=SPEC_POLY_ORDER, polynomial_mode="marginalize",
                  polynomial_prior_sigma=0.1)
    return Spectrum(**kw)


def default_spec_wave(zred, R=SPEC_R):
    # rest 3500-7000 A redshifted, two pixels per resolution element (FWHM)
    lo, hi = 3500.0 * (1 + zred), 7000.0 * (1 + zred)
    n = int(np.log(hi / lo) * 2 * R)
    return np.geomspace(lo, hi, n)

## B5. Priors: which parameters are free and which ones are fixed?

Every parameter the model samples needs a prior. `prior_for(name)` holds a sensible default for
each name this notebook can create; **edit it** to change a prior. Every CSP parameter without a
prior is held **fixed** at its default with a constant transform, but the next cell will print a summary of what is fixed and what is free,
so nobody gets confused.

Part A fixed the nebular parameters with `CSPBasis(fixed=...)`, which you choose before the CSP
exists. Here the knobs decide which parameters exist, so the next cell fixes everything without a
prior with a constant transform, `transforms[name] = lambda th: value`, after the CSP is built. The
two are the same model: ceridwen turns each `fixed=` entry into exactly that constant transform.

In [ ]:
def prior_for(name):
    lz_lo = -1.5 if GAS_TIED else -2.0                       # gas tied: stay inside the CLOUDY axis
    table = {
        "logmass":            Uniform(low=6.0, high=12.5),
        "logsfr_ratios":      StudentT(mean=0.0, scale=SFH_PRIOR_SCALE, df=2.0),
        "logzsol":            Uniform(low=lz_lo, high=0.2),
        "logzsol_bins":       Uniform(low=lz_lo, high=0.2),
        "afe":                Uniform(low=-0.2, high=0.6),
        "gas_logz":           Uniform(low=-1.5, high=0.2),
        "gas_logu":           Uniform(low=-3.5, high=-1.0),
        "tau_pow":            ClippedNormal(mean=0.3, sigma=0.5, low=0.0, high=4.0),
        "alpha_pow":          ClippedNormal(mean=-1.0, sigma=0.5, low=-2.5, high=0.5),
        "diffuse_dust_index": Uniform(low=-1.0, high=0.4),
        "duste_qpah":         Uniform(low=0.5, high=4.5),       # DL07 axis 0.47-4.58
        "duste_umin":         Uniform(low=0.1, high=25.0),      # DL07 axis 0.1-25
        "duste_gamma":        Uniform(low=0.001, high=0.15),
        "frac_obrun":         Uniform(low=0.0, high=1.0),
        "igm_factor":         Uniform(low=0.0, high=2.0),
        "log_fagn":           Uniform(low=-5.0, high=float(np.log10(3.0))),
        "log_fagn_bbb":       Uniform(low=-5.0, high=2.0),
        "agn_tau":            LogUniform(mini=5.0, maxi=150.0),
        "zred":               Uniform(low=ZRED - 0.1, high=ZRED + 0.1),
        "sigma_gal":          Uniform(low=20.0, high=600.0),
        "sigma_gas":          Uniform(low=20.0, high=600.0),
        "lsf_scale":          Uniform(low=0.8, high=1.3),
        "eline_scaling":      Uniform(low=0.1, high=2.0),
        "spectrum_scaling":   ClippedNormal(mean=1.0, sigma=0.3, low=0.2, high=3.0),
        "spectrum_calib":     Uniform(low=-0.2, high=0.2),
        "f_outlier_phot":     Uniform(low=0.0, high=0.5),
        "f_outlier_spec":     Uniform(low=0.0, high=0.5),
    }
    if name in table:
        return table[name]
    if name.startswith("diffuse_tau"):                          # the optical depth of any diffuse law
        return ClippedNormal(mean=0.3, sigma=1.0, low=0.0, high=4.0)
    return None


# the mock galaxy's input values (also the start values of the free parameters)
TRUTH_VALUES = {"logmass": 9.5, "logzsol": -0.5, "afe": 0.2, "gas_logz": -0.5, "gas_logu": -2.3,
                "tau_pow": 0.3, "alpha_pow": -1.0, "diffuse_dust_index": -0.2,
                "duste_qpah": 2.0, "duste_umin": 1.0, "duste_gamma": 0.01, "frac_obrun": 0.1,
                "igm_factor": 1.0, "log_fagn": -2.0, "log_fagn_bbb": -2.0, "agn_tau": 20.0,
                "zred": ZRED, "sigma_gal": 150.0, "sigma_gas": 150.0, "lsf_scale": 1.0,
                "eline_scaling": 1.0, "spectrum_scaling": 1.0,
                "f_outlier_phot": 0.0, "f_outlier_spec": 0.0}

In [ ]:
times_B = np.array(csp_B.sfh_times)
TRANSFORMS_B = {"sfh": lambda th, _t=times_B: logsfr_ratios_to_sfh(th["logsfr_ratios"], sfh_times_yr=_t)}
PRIORS_B, INIT_B, TRUTH_B, FIXED_B = {}, {}, {}, {}

def free(name, init):
    # sample ``name``: its prior from prior_for, its start value ``init``
    p = prior_for(name)
    if p is None:
        raise KeyError(f"no prior for {name!r}: add it to prior_for")
    PRIORS_B[name] = p
    INIT_B[name] = jnp.atleast_1d(jnp.asarray(init, dtype=float))

# the SFH and the mass are always free
free("logsfr_ratios", np.zeros(N_RATIOS))
free("logmass", TRUTH_VALUES["logmass"])
TRUTH_B["logsfr_ratios"] = jnp.asarray(np.linspace(0.3, -0.3, N_RATIOS))

# AGN amplitudes are sampled in log10 and transformed
if AGN:
    for k in ("fagn", "fagn_bbb"):
        if k in csp_B.param_names:          # agn_mode="fsps" has no disc, so no fagn_bbb
            TRANSFORMS_B[k] = (lambda key: (lambda th: 10.0 ** th[key]))(f"log_{k}")
            free(f"log_{k}", TRUTH_VALUES[f"log_{k}"])

# every CSP parameter: free with its prior, or fixed at its default
for name in csp_B.param_names:
    if name in ("sfh",) or name in TRANSFORMS_B:
        continue
    p = prior_for(name)
    if p is None:
        v = jnp.asarray(csp_B.theta_init[name])
        TRANSFORMS_B[name] = (lambda val: (lambda th: val))(v)
        FIXED_B[name] = np.asarray(v)
    else:
        default = csp_B.theta_init[name]
        start = TRUTH_VALUES.get(name, None)
        if name.startswith("diffuse_tau"):
            start = 0.3
        if name == "logzsol_bins":
            start = np.full(np.shape(default), TRUTH_VALUES["logzsol"])
        free(name, default if start is None else np.broadcast_to(start, np.shape(default)))

# model-level options (not CSP defaults: each is switched on by sampling it)
if IS_AFE and FIT_AFE:
    free("afe", TRUTH_VALUES["afe"])        # without it the CSP uses [alpha/Fe] = 0
if FIT_ZRED:
    free("zred", ZRED)
if FIT_FESC:
    free("frac_obrun", TRUTH_VALUES["frac_obrun"])
if FIT_IGM_FACTOR and IGM:
    free("igm_factor", 1.0)
if SIGMA_GAL == "free":
    free("sigma_gal", TRUTH_VALUES["sigma_gal"])
if SIGMA_GAS == "free":
    free("sigma_gas", TRUTH_VALUES["sigma_gas"])
gal = "sigma_gal" if SIGMA_GAL == "free" else float(SIGMA_GAL)
gas = TIED if SIGMA_GAS == "tied" else ("sigma_gas" if SIGMA_GAS == "free" else float(SIGMA_GAS))
KIN_B = Kinematics(sigma_gal=gal, sigma_gas=gas)

# observation nuisance terms
if USE_PHOTOMETRY and PHOT_OUTLIERS:
    free("f_outlier_phot", 0.01)
if USE_LINES and ELINE_SCALING:
    free("eline_scaling", 1.0)
if USE_SPECTRUM:
    if FIT_LSF_SCALE:
        free("lsf_scale", 1.0)
    if SPEC_CALIBRATION in ("scaling", "scaling+legendre"):
        free("spectrum_scaling", 1.0)
    if SPEC_CALIBRATION == "scaling+legendre":
        free("spectrum_calib", np.zeros(SPEC_POLY_ORDER))
    if SPEC_OUTLIERS:
        free("f_outlier_spec", 0.01)

for k, v in INIT_B.items():                 # the mock truth = the start values, except the SFH
    TRUTH_B.setdefault(k, v)

print("free :", {k: tuple(np.shape(v)) for k, v in INIT_B.items()})
print("fixed:", {k: np.round(v, 3).tolist() for k, v in FIXED_B.items()})

## B6. The data and the model

For a mock, the truth goes through the same model and gets Gaussian noise.
The spectrum jitter prior is set relative to the spectrum's own errors.

In [ ]:
rng = np.random.default_rng(2)

def noisy(v, snr):
    v = np.asarray(v, float)
    err = np.maximum(np.abs(v) / snr, np.median(np.abs(v)) / (3 * snr))
    return v + err * rng.standard_normal(v.size), err

if DATA_SOURCE == "mock":
    shells = []
    if USE_PHOTOMETRY:
        nf = len(DEFAULT_FILTERS)
        shells.append(make_photometry(DEFAULT_FILTERS, np.zeros(nf), np.ones(nf)))
    rows = LINE_ROWS or (default_line_rows() if USE_LINES else [])
    if USE_LINES:
        shells.append(make_lines(rows, np.zeros(len(rows)), np.ones(len(rows))))
    if USE_SPECTRUM:
        w = default_spec_wave(ZRED)
        shells.append(make_spectrum(w, np.zeros(w.size), np.ones(w.size)))
    truth_model_B = SedModel(csp_B, shells, priors=PRIORS_B, transforms=TRANSFORMS_B,
                             free_param_init=TRUTH_B, zred=ZRED, kinematics=KIN_B,
                             broaden_photometry=BROADEN_PHOTOMETRY)
    true_B = truth_model_B.predict(truth_model_B.theta_init)
    OBS_B = []
    if USE_PHOTOMETRY:
        f, e = noisy(true_B["phot"], 15.0)
        OBS_B.append(make_photometry(DEFAULT_FILTERS, f, e))
    if USE_LINES:
        f, e = noisy(true_B["lines"], 10.0)
        OBS_B.append(make_lines(rows, f, e))
    if USE_SPECTRUM:
        f, e = noisy(true_B["spec"], 10.0)
        OBS_B.append(make_spectrum(w, f, e))
else:
    OBS_B = []
    if USE_PHOTOMETRY:
        OBS_B.append(make_photometry(MY_FILTERS, MY_PHOT_NJY / NJY, MY_PHOT_ERR / NJY))
    if USE_LINES:
        OBS_B.append(make_lines(MY_LINE_ROWS, MY_LINE_FLUX, MY_LINE_ERR))
    if USE_SPECTRUM:
        OBS_B.append(make_spectrum(MY_SPEC_WAVE, MY_SPEC_NJY * NJY_TO_CGS, MY_SPEC_ERR * NJY_TO_CGS))

if USE_SPECTRUM and SPEC_JITTER:
    med = float(np.median(np.asarray([o for o in OBS_B if o.name == "spec"][0].uncertainty)))
    PRIORS_B["log_jitter_spec"] = Uniform(low=float(np.log(0.01 * med)), high=float(np.log(2 * med)))
    INIT_B["log_jitter_spec"] = jnp.array([float(np.log(0.1 * med))])

for o in OBS_B:
    print(o)

**The model.** `SedModel` joins the physics, the data, the priors and the transforms.
`model.display()` draws each free parameter with its prior, the stage of the forward model it
enters, and the range of data the priors allow. `model.latex()` gives a table of every free,
derived and fixed parameter for a paper.

In [ ]:
model_B = SedModel(csp_B, OBS_B, priors=PRIORS_B, transforms=TRANSFORMS_B,
                   free_param_init=INIT_B, zred=ZRED, kinematics=KIN_B,
                   broaden_photometry=BROADEN_PHOTOMETRY)
print(model_B.summary())
model_B.display()

In [ ]:
print(model_B.latex())

## B7. Fit

`fitSED` builds one likelihood per observation (with the noise terms you sampled), runs the
sampler and writes `fit_partB/ceridwen_result.h5`.

| setting | meaning | demo | science |
|---|---|---|---|
| `num_live` | live points | 200 | ≥ 500 |
| `num_delete` | replaced per iteration, in parallel on the GPU | 40 | ~ num_live / 5 |
| `num_inner_steps` | slice steps per replacement | 30 | ≥ 30; float32 evidences need more (docs/mac_gpu.md) |
| `logZ_tol` | stop when the evidence left in the live points is below e^tol | -5 | -5 |

`sampler="nuts"` runs Hamiltonian Monte Carlo instead (no evidence; `docs/samplers.md`).

In [ ]:
MFRAC = False if IS_AFE else None   # None: write the surviving mass when the grid has the table
t0 = time.time()
result_B = fitSED(model_B, sampler="nested", rng_key=jax.random.PRNGKey(0),
                  sampler_kwargs={"num_live": 200, "num_delete": 40, "num_inner_steps": 30,
                                  "logZ_tol": -5.0},
                  output_dir="fit_partB", mfrac=MFRAC)
print(f"ln Z = {result_B.log_evidence:.2f} +- {result_B.log_evidence_err:.2f}, "
      f"{result_B.n_likelihood_calls:,} likelihood calls, {time.time() - t0:.0f} s")

## B8. Post-processing

`PostProcess(model, result)` takes a `SamplingResult` or the path of a result file (so you can
come back to a fit later with the same model). `windows_myr` sets the SFR averages it computes.
A custom quantity is a function of one draw (`SpectrumSample`); here A_V from the dust-free and
the full spectrum.

In [ ]:
def A_V(s):
    i = s.index_of(5500.0)
    return 2.5 * np.log10(s.dustfree[i] / s.full[i])

pp_B = PostProcess(model_B, result_B, n_samples=1000, windows_myr=(10.0, 50.0, 100.0),
                   mfrac=MFRAC, derived={"A_V": A_V})
out_B = pp_B.run()

def q(x):
    lo, med, hi = np.percentile(np.ravel(x), [16, 50, 84])
    return f"{med:+.3g} (-{med - lo:.2g}/+{hi - med:.2g})"

for k in INIT_B:
    if np.size(INIT_B[k]) == 1:
        t = f"   truth {float(np.ravel(TRUTH_B[k])[0]):+.3g}" if DATA_SOURCE == "mock" else ""
        print(f"{k:>20}: {q(out_B['theta'][k])}{t}")
s = out_B["extras"]["sfh"]
print(f"{'log SFR_10':>20}: {q(np.log10(s['sfr10']))}")
print(f"{'b = log SFR10/SFR10-50':>20}: {q(np.log10(s['sfr10'] / ((50 * s['sfr50'] - 10 * s['sfr10']) / 40)))}")
print(f"{'M_UV':>20}: {q(out_B['extras']['uv']['MUV'])}")
print(f"{'A_V':>20}: {q(out_B['derived']['A_V'])}")
if "mass_surviving" in s:
    print(f"{'log M_surviving':>20}: {q(np.log10(s['mass_surviving']))}")

In [ ]:
truths_B = ({k: (float(np.ravel(v)[0]) if np.size(v) == 1 else np.asarray(v)) for k, v in TRUTH_B.items()
             if k in INIT_B} if DATA_SOURCE == "mock" else None)
crop_sed(summary_figure(out_B, model_B, title="Part B", truths=truths_B), model_B); plt.show()
corner_figure(out_B, truths=truths_B); plt.show()

**Saving and coming back.** `pp.figures` writes the summary, corner and sampler-diagnostic
PDFs (read `diagnostics.pdf` before trusting a fit); `pp.save` writes every draw to an `.npz`
that `load_postprocess` reads back; a new `PostProcess` can be built from the result file.

In [ ]:
paths = pp_B.figures("fit_partB/figures", title="Part B")
print(paths)
pp_B.save("fit_partB/post.npz")
again = load_postprocess("fit_partB/post.npz")
print(sorted(again))
pp_again = PostProcess(model_B, "fit_partB/ceridwen_result.h5", n_samples=500, mfrac=MFRAC)

## Where next

- Data files: `np.genfromtxt("my.csv", delimiter=",", names=True)` gives the columns by name;
  convert nJy with `/ NJY` (photometry) and `* NJY_TO_CGS` (spectrum).
- `docs/` on the website: tutorial, samplers, post-processing, noise and calibration,
  emission-line marginalisation, α/Fe, AGN, conventions.
- `GOTCHAS.md`: every known trap and its guard.
- `examples/`: `demo_3_spectrum_advanced.py` (spectrum + photometry), `agn_demo.py`,
  `demo_afe_quiescent.py`, `demo_eline_marginalisation.py`.